<a href="https://colab.research.google.com/github/rushitpatel2311/deepfack_groupproject/blob/rushit/Group_7_Deepfake_Detection_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎭 Group 7 — AI-Driven Deepfake Detection System
**Module:** AI Systems Engineering (CMP-L044) — Part 2 Artefact  
**Programme:** MSc Artificial Intelligence, University of Roehampton  

---

## Notebook Structure

| Section | Owner | Description |
|---------|-------|-------------|
| **Section 0** | All | Environment setup, dataset config |
| **Section 1** | Isha Luhar | Data ingestion, preprocessing pipeline |
| **Section 2** | Nishtha Solanki | Video inference pipeline (EfficientNet-B4 + ViT-Small) |
| **Section 3** | OM Mistry | Audio inference pipeline (ResNet-18 on Mel spectrograms) |
| **Section 4** | Rushitkumar Patel | Late fusion meta-learner + Grad-CAM explainability |
| **Section 5** | All | Evaluation, fairness analysis, MLflow tracking |

---

## ⚠️ IMPORTANT: Read Before Running

1. **Runtime → Change runtime type → T4 GPU** (free tier is sufficient)
2. Run cells **top to bottom** — sections depend on each other
3. Dataset setup instructions are in **Section 0, Cell 0.2**
4. If Colab disconnects, re-run from Section 0 (packages reset)

---

## 🔗 How to Open This Notebook from GitHub (Colab built-in)

> **Every group member** follows these steps — no tokens, no code needed:

1. Go to **[colab.research.google.com](https://colab.research.google.com)**
2. Click **File → Open notebook**
3. Select the **GitHub** tab
4. Sign in with your GitHub account when prompted
5. Type your repo name (e.g. )
6. Select **** from the list
7. Colab opens it directly — click **▶ Run All** or run cells top to bottom

## 💾 How to Save Changes Back to GitHub

1. After making changes in Colab, click **File → Save a copy in GitHub**
2. Select your repo and branch ()
3. Write a commit message (e.g. )
4. Click **OK** — Colab commits directly, no terminal needed

---

## Declaration of Use of Generative AI
Generative AI tools (ChatGPT, Claude, Google Gemini) were used for idea generation, code structuring suggestions, and summarising research papers. All outputs were reviewed and substantially rewritten by group members. Full declaration is in the report.


---
# SECTION 0 — Environment Setup
**All members**

Install all dependencies and configure dataset path. Run every time Colab restarts.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 0.1 — Install all dependencies
# ─────────────────────────────────────────────────────────────────────────────
print("Installing dependencies...")

!pip install -q \
    timm \
    facenet \
    librosa \
    mlflow \
    grad-cam \
    scikit-learn \
    opencv-python-headless \
    albumentations \
    matplotlib \
    seaborn \
    tqdm \
    pandas \
    Pillow \
    scipy \

print("✅ All packages installed.")

Installing dependencies...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 48.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 88.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 116.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 112.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 83.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

> **ℹ️ No GitHub code needed here.**  
> Open this notebook via **File → Open notebook → GitHub tab** in Colab.  
> Save changes back via **File → Save a copy in GitHub**.  
> That is all — proceed to Cell 0.2 below to configure your dataset.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 0.2 — Dataset Setup
# ─────────────────────────────────────────────────────────────────────────────
#
# ════════════════════════════════════════════════════════════════════
#  HOW TO ADD YOUR DATASETS (choose ONE method per dataset)
# ════════════════════════════════════════════════════════════════════
#
#  METHOD 1 — Google Drive Mount (Recommended for large datasets)
#  ---------------------------------------------------------------
#  a) Upload your dataset ZIP to Google Drive
#  b) Uncomment the Drive mount block below
#  c) Update the DRIVE_DATASET_PATH to your Drive path
#
#  METHOD 2 — Kaggle API (For FaceForensics++, DFDC subsets)
#  ----------------------------------------------------------
#  a) Go to kaggle.com → Account → Create API Token → download kaggle.json
#  b) In Colab Secrets, add: KAGGLE_USERNAME and KAGGLE_KEY
#  c) Uncomment the Kaggle block below
#
#  METHOD 3 — Direct Upload (Small samples only)
#  -----------------------------------------------
#  Use Colab left panel → Files → Upload
#  Then update DATA_ROOT below to point to your uploaded folder
#
#  ════════════════════════════════════════════════════════════════════
#  EXPECTED FOLDER STRUCTURE (create this in Drive or after unzipping):
#
#  data/
#  ├── train/
#  │   ├── real/          ← real video frames (.jpg or .png)
#  │   └── fake/          ← deepfake video frames (.jpg or .png)
#  ├── val/
#  │   ├── real/
#  │   └── fake/
#  ├── test/
#  │   ├── real/
#  │   └── fake/
#  └── audio/
#      ├── real/          ← real audio clips (.wav)
#      └── fake/          ← fake/spoofed audio clips (.wav)
#
#  ════════════════════════════════════════════════════════════════════
#  RECOMMENDED FREE DATASETS (small subsets for academic use):
#  - FaceForensics++ sample: https://github.com/ondyari/FaceForensics
#  - DFDC preview: https://ai.facebook.com/datasets/dfdc/
#  - ASVspoof 2019 LA: https://datashare.ed.ac.uk/handle/10283/3336
#  ════════════════════════════════════════════════════════════════════

import os

# ─── METHOD 1: Google Drive ───────────────────────────────────────────
# from google.colab import drive
# drive.mount('/content/drive')
# DRIVE_DATASET_PATH = '/content/drive/MyDrive/deepfake_data'  # ← update this
# !cp -r {DRIVE_DATASET_PATH} /content/data

# ─── METHOD 2: Kaggle ─────────────────────────────────────────────────
# from google.colab import userdata
# os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
# os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
# !pip install -q kaggle
# !kaggle datasets download -d <dataset-slug> -p /content/data --unzip

# ─── DATA ROOT (update if using Drive or different path) ──────────────
DATA_ROOT = "/content/data"  # ← change this if your data is elsewhere

# Create placeholder structure if data not yet uploaded (runs in demo mode)
for split in ['train', 'val', 'test']:
    for label in ['real', 'fake']:
        os.makedirs(f"{DATA_ROOT}/{split}/{label}", exist_ok=True)
os.makedirs(f"{DATA_ROOT}/audio/real", exist_ok=True)
os.makedirs(f"{DATA_ROOT}/audio/fake", exist_ok=True)

# Check what's available
print("📊 Dataset status:")
for split in ['train', 'val', 'test']:
    real_count = len([f for f in os.listdir(f"{DATA_ROOT}/{split}/real") if f.endswith(('.jpg','.png'))])
    fake_count = len([f for f in os.listdir(f"{DATA_ROOT}/{split}/fake") if f.endswith(('.jpg','.png'))])
    print(f"  {split:5s} — real: {real_count:4d} | fake: {fake_count:4d}")

if real_count == 0:
    print("\n⚠️  No images found — running in DEMO MODE with synthetic data.")
    print("   Add real dataset images to /content/data/ to train properly.")
    DEMO_MODE = True
else:
    DEMO_MODE = False
    print("\n✅ Dataset found. DEMO_MODE = False")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 0.3 — Global Imports & Configuration
# ─────────────────────────────────────────────────────────────────────────────
import os, random, json, warnings, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models

import timm
import librosa
import librosa.display
import cv2
import mlflow
import mlflow.pytorch
from sklearn.metrics import (
    roc_auc_score, accuracy_score, confusion_matrix,
    classification_report, roc_curve
)
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV

warnings.filterwarnings('ignore')

# ─── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

# ─── Device ───────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️  Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ─── Global Config ────────────────────────────────────────────────────────────
CFG = {
    # Data
    'data_root'     : DATA_ROOT,
    'img_size'      : 224,
    'n_frames'      : 16,          # frames per video clip for temporal model
    'sample_rate'   : 16000,       # audio sample rate
    'n_mels'        : 128,         # mel spectrogram bins
    'hop_length'    : 160,         # 10ms hop @ 16kHz
    'win_length'    : 400,         # 25ms window
    # Training
    'batch_size'    : 16,
    'epochs_video'  : 5,
    'epochs_audio'  : 5,
    'lr'            : 1e-4,
    'weight_decay'  : 1e-4,
    # MLflow
    'experiment'    : 'Group7_Deepfake_Detection',
    # Thresholds (from Part 1 NFRs)
    'latency_ms'    : 500,
    'auc_target'    : 0.92,
    'fpr_target'    : 0.03,
    'fairness_target': 0.05,
}

# ─── MLflow Setup ─────────────────────────────────────────────────────────────
mlflow.set_experiment(CFG['experiment'])
print(f"\n📊 MLflow experiment: {CFG['experiment']}")
print("\n✅ Configuration complete. All global settings loaded.")

---
> **💾 Remember:** After finishing your section, go to  
> **File → Save a copy in GitHub** to commit your progress.  
> Write a clear message e.g. *Section 2 done — video model training complete*.  
> Do this after every significant change so your commit history shows individual contributions.


---
# SECTION 1 — Data Ingestion & Preprocessing Pipeline
**Lead: Isha Luhar (A00085061)**

Implements FR1 and FR2 from Part 1: face detection via MTCNN, frame extraction at 10fps,
image normalisation to 224×224, and Mel spectrogram generation for audio.
All preprocessing configs are logged to MLflow for full reproducibility (NFR6).

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 1.1 — Face Detector (MTCNN) + Frame Preprocessing
# Lead: Isha Luhar
# ─────────────────────────────────────────────────────────────────────────────
from facenet_pytorch import MTCNN

class FaceDetector:
    """
    MTCNN-based face detector.
    Returns cropped, aligned face crops at target size.
    Falls back to centre-crop if no face detected (graceful degradation).
    """
    def __init__(self, image_size=224, device=DEVICE):
        self.image_size = image_size
        self.detector = MTCNN(
            image_size=image_size,
            margin=20,
            min_face_size=40,
            thresholds=[0.6, 0.7, 0.7],
            factor=0.709,
            post_process=True,
            keep_all=False,     # return highest-confidence face only
            device=device
        )
        self.fallback_transform = T.Compose([
            T.Resize((image_size, image_size)),
            T.ToTensor(),
        ])

    def detect(self, pil_image):
        """
        Args: PIL Image
        Returns: torch.Tensor [3, H, W] — face crop or centre-crop fallback
        """
        face_tensor = self.detector(pil_image)
        if face_tensor is None:
            # Graceful fallback — no face found, use resized full frame
            return self.fallback_transform(pil_image)
        return face_tensor  # already [3, H, W] normalised by MTCNN

    def detect_from_path(self, image_path):
        img = Image.open(image_path).convert('RGB')
        return self.detect(img)


# ─── Test face detector on a dummy image ─────────────────────────────────────
face_detector = FaceDetector(image_size=CFG['img_size'], device=DEVICE)
dummy_img = Image.fromarray(np.random.randint(0, 255, (480, 640, 3), dtype=np.uint8))
face_tensor = face_detector.detect(dummy_img)
print(f"✅ FaceDetector initialised.")
print(f"   Output tensor shape: {face_tensor.shape}  (expected [3, 224, 224])")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 1.2 — Image Dataset with Augmentation
# Lead: Isha Luhar
# ─────────────────────────────────────────────────────────────────────────────
import albumentations as A
from albumentations.pytorch import ToTensorV2

def get_video_transforms(split='train'):
    """
    Returns albumentations transforms.
    Train: augmentation (compression sim, noise, flips, jitter)
    Val/Test: only resize + normalise
    """
    mean = [0.485, 0.456, 0.406]  # ImageNet stats
    std  = [0.229, 0.224, 0.225]

    if split == 'train':
        return A.Compose([
            A.Resize(CFG['img_size'], CFG['img_size']),
            A.HorizontalFlip(p=0.5),
            A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, p=0.5),
            A.GaussNoise(var_limit=(10, 50), p=0.3),          # simulate noise
            A.ImageCompression(quality_lower=60, quality_upper=100, p=0.4),  # compression sim
            A.Rotate(limit=10, p=0.3),
            A.Normalize(mean=mean, std=std),
            ToTensorV2(),
        ])
    else:
        return A.Compose([
            A.Resize(CFG['img_size'], CFG['img_size']),
            A.Normalize(mean=mean, std=std),
            ToTensorV2(),
        ])


class DeepfakeFrameDataset(Dataset):
    """
    PyTorch Dataset for deepfake frame classification.
    Loads image frames from data/train|val|test/real|fake/
    Labels: 0 = real, 1 = fake
    """
    def __init__(self, root, split='train', transform=None, demo_mode=False):
        self.transform  = transform or get_video_transforms(split)
        self.demo_mode  = demo_mode
        self.samples    = []
        self.labels     = []

        if demo_mode:
            # Generate synthetic data for testing pipeline without real dataset
            n = 200 if split == 'train' else 50
            self.samples = [None] * n
            self.labels  = [random.randint(0, 1) for _ in range(n)]
            print(f"   [DEMO] {split}: {n} synthetic samples")
            return

        for label_name, label_idx in [('real', 0), ('fake', 1)]:
            folder = Path(root) / split / label_name
            if not folder.exists():
                continue
            files = [f for f in folder.iterdir()
                     if f.suffix.lower() in ('.jpg', '.jpeg', '.png')]
            self.samples.extend(files)
            self.labels.extend([label_idx] * len(files))

        print(f"   {split}: {len(self.samples)} samples "
              f"(real={self.labels.count(0)}, fake={self.labels.count(1)})")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        label = self.labels[idx]
        if self.demo_mode or self.samples[idx] is None:
            img = np.random.randint(0, 255, (CFG['img_size'], CFG['img_size'], 3), dtype=np.uint8)
        else:
            img = np.array(Image.open(self.samples[idx]).convert('RGB'))

        if self.transform:
            img = self.transform(image=img)['image']
        return img, torch.tensor(label, dtype=torch.long)


# Build datasets and loaders
print("📂 Loading datasets:")
train_ds = DeepfakeFrameDataset(DATA_ROOT, 'train', demo_mode=DEMO_MODE)
val_ds   = DeepfakeFrameDataset(DATA_ROOT, 'val',   demo_mode=DEMO_MODE)
test_ds  = DeepfakeFrameDataset(DATA_ROOT, 'test',  demo_mode=DEMO_MODE)

train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=CFG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)

print(f"\n✅ DataLoaders ready — batch size: {CFG['batch_size']}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 1.3 — Mel Spectrogram Generator (Audio Preprocessing)
# Lead: Isha Luhar
# ─────────────────────────────────────────────────────────────────────────────

class MelSpectrogramGenerator:
    """
    Converts raw audio (.wav) to 128-bin Mel spectrograms.
    Config follows Part 1: 25ms window, 10ms hop, 16kHz SR.
    """
    def __init__(self, sr=16000, n_mels=128, hop_length=160, win_length=400,
                 n_fft=512, fmin=20, fmax=8000, duration=3.0):
        self.sr          = sr
        self.n_mels      = n_mels
        self.hop_length  = hop_length
        self.win_length  = win_length
        self.n_fft       = n_fft
        self.fmin        = fmin
        self.fmax        = fmax
        self.duration    = duration
        self.target_len  = int(sr * duration)

    def wav_to_spectrogram(self, wav_path=None, y=None):
        """
        Args:
            wav_path: path to .wav file
            y: pre-loaded numpy waveform (optional)
        Returns:
            spec: np.ndarray [n_mels, time_frames] (log-scale)
        """
        if y is None:
            y, _ = librosa.load(wav_path, sr=self.sr, mono=True)

        # Pad or trim to fixed duration
        if len(y) < self.target_len:
            y = np.pad(y, (0, self.target_len - len(y)))
        else:
            y = y[:self.target_len]

        mel = librosa.feature.melspectrogram(
            y=y, sr=self.sr,
            n_fft=self.n_fft,
            hop_length=self.hop_length,
            win_length=self.win_length,
            n_mels=self.n_mels,
            fmin=self.fmin,
            fmax=self.fmax
        )
        log_mel = librosa.power_to_db(mel, ref=np.max)  # log scale
        return log_mel

    def to_tensor(self, spec):
        """ Normalise and convert to [1, n_mels, time] tensor. """
        spec_norm = (spec - spec.mean()) / (spec.std() + 1e-6)
        return torch.tensor(spec_norm, dtype=torch.float32).unsqueeze(0)

    def visualise(self, wav_path=None, y=None, sr=None, title="Mel Spectrogram"):
        spec = self.wav_to_spectrogram(wav_path=wav_path, y=y)
        fig, ax = plt.subplots(figsize=(10, 4))
        img = librosa.display.specshow(
            spec, sr=self.sr, hop_length=self.hop_length,
            x_axis='time', y_axis='mel', ax=ax, fmin=self.fmin, fmax=self.fmax
        )
        fig.colorbar(img, ax=ax, format='%+2.0f dB')
        ax.set_title(title)
        plt.tight_layout()
        plt.show()
        return spec


mel_gen = MelSpectrogramGenerator(
    sr=CFG['sample_rate'], n_mels=CFG['n_mels'],
    hop_length=CFG['hop_length'], win_length=CFG['win_length']
)

# Demo: generate and visualise a synthetic spectrogram
dummy_audio = np.random.randn(CFG['sample_rate'] * 3).astype(np.float32)
dummy_spec  = mel_gen.wav_to_spectrogram(y=dummy_audio)
print(f"✅ Mel spectrogram shape: {dummy_spec.shape}  (expected [{CFG['n_mels']}, ~300])")
mel_gen.visualise(y=dummy_audio, title="Demo Mel Spectrogram (synthetic audio)")

---
# SECTION 2 — Video Inference Pipeline
**Lead: Nishtha Solanki (A00087199)**

Implements the hybrid CNN-Transformer video pipeline from Part 1 Section 4:
- **EfficientNet-B4** spatial feature extractor (1792-dim embeddings per frame)
- **ViT-Small** temporal attention over 16-frame windows
- Training loop with MLflow experiment tracking

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2.1 — EfficientNet-B4 Spatial Feature Extractor
# Lead: Nishtha Solanki
# ─────────────────────────────────────────────────────────────────────────────

class EfficientNetFeatureExtractor(nn.Module):
    """
    EfficientNet-B4 spatial feature extractor.
    Removes classification head — returns 1792-dim embedding per frame.
    Pre-trained on ImageNet; fine-tuned on deepfake data.

    Justification: Wang et al. [16] showed CNN-transformer hybrid achieves
    within 2.1% AUC of full transformer while reducing inference time by 61%.
    EfficientNet-B4 chosen for its accuracy-efficiency profile (Part 1 Section 4).
    """
    def __init__(self, pretrained=True, freeze_base=False):
        super().__init__()
        self.backbone = timm.create_model(
            'efficientnet_b4',
            pretrained=pretrained,
            num_classes=0,          # remove head
            global_pool='avg'
        )
        self.feature_dim = self.backbone.num_features  # 1792

        if freeze_base:
            # Freeze early layers — fine-tune only top layers
            for name, param in self.backbone.named_parameters():
                if 'blocks.5' not in name and 'blocks.6' not in name:
                    param.requires_grad = False

    def forward(self, x):
        """ x: [B, 3, H, W] → [B, 1792] """
        return self.backbone(x)


# ─── Sanity check ─────────────────────────────────────────────────────────────
efficientnet = EfficientNetFeatureExtractor(pretrained=True, freeze_base=True).to(DEVICE)
dummy_frames = torch.randn(4, 3, 224, 224).to(DEVICE)
with torch.no_grad():
    feats = efficientnet(dummy_frames)
print(f"✅ EfficientNet-B4 loaded.")
print(f"   Input:  {dummy_frames.shape}")
print(f"   Output: {feats.shape}  (expected [4, 1792])")
n_params = sum(p.numel() for p in efficientnet.parameters())
n_train  = sum(p.numel() for p in efficientnet.parameters() if p.requires_grad)
print(f"   Total params: {n_params/1e6:.1f}M | Trainable: {n_train/1e6:.1f}M")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2.2 — Hybrid CNN-Transformer Video Classifier
# Lead: Nishtha Solanki
# ─────────────────────────────────────────────────────────────────────────────

class VideoDeepfakeClassifier(nn.Module):
    """
    Hybrid EfficientNet-B4 + ViT-Small architecture.

    For frame-level input (single images from our dataset):
      EfficientNet-B4 → projection → binary classification head

    The ViT temporal component is activated when processing clip sequences.
    This design matches Wang et al. [16] M2TR and Part 1 Section 4 description.
    """
    def __init__(self, feature_dim=1792, proj_dim=512, num_classes=2,
                 dropout=0.3, use_temporal=False):
        super().__init__()
        self.use_temporal = use_temporal

        # Spatial feature extractor
        self.spatial = EfficientNetFeatureExtractor(pretrained=True, freeze_base=True)

        # Projection layer
        self.projector = nn.Sequential(
            nn.Linear(feature_dim, proj_dim),
            nn.LayerNorm(proj_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        if use_temporal:
            # Lightweight transformer encoder for temporal modelling
            encoder_layer = nn.TransformerEncoderLayer(
                d_model=proj_dim, nhead=8, dim_feedforward=1024,
                dropout=dropout, batch_first=True
            )
            self.temporal = nn.TransformerEncoder(encoder_layer, num_layers=4)

        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(proj_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        """
        Frame mode: x [B, 3, H, W]
        Clip mode:  x [B, T, 3, H, W] (when use_temporal=True)
        Returns: logits [B, 2], features [B, proj_dim]
        """
        if self.use_temporal and x.dim() == 5:
            B, T, C, H, W = x.shape
            x_flat = x.view(B * T, C, H, W)
            feats   = self.spatial(x_flat).view(B, T, -1)
            proj    = self.projector(feats)           # [B, T, proj_dim]
            temporal= self.temporal(proj)             # [B, T, proj_dim]
            pooled  = temporal.mean(dim=1)            # [B, proj_dim]
        else:
            feats  = self.spatial(x)                  # [B, 1792]
            pooled = self.projector(feats)             # [B, proj_dim]

        logits = self.classifier(pooled)
        return logits, pooled

    def predict_proba(self, x):
        logits, _ = self.forward(x)
        return F.softmax(logits, dim=-1)[:, 1]  # P(fake)


video_model = VideoDeepfakeClassifier(use_temporal=False).to(DEVICE)
with torch.no_grad():
    logits, feats = video_model(dummy_frames)
print(f"✅ VideoDeepfakeClassifier built.")
print(f"   Logits shape:   {logits.shape}  (expected [4, 2])")
print(f"   Features shape: {feats.shape}   (expected [4, 512])")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2.3 — Training Loop (Video Model)
# Lead: Nishtha Solanki
# ─────────────────────────────────────────────────────────────────────────────

def train_one_epoch(model, loader, optimiser, criterion, device):
    model.train()
    total_loss, total_correct, total_samples = 0., 0, 0
    for imgs, labels in tqdm(loader, desc='  Train', leave=False):
        imgs, labels = imgs.to(device), labels.to(device)
        optimiser.zero_grad()
        logits, _ = model(imgs)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimiser.step()
        total_loss    += loss.item() * imgs.size(0)
        total_correct += (logits.argmax(1) == labels).sum().item()
        total_samples += imgs.size(0)
    return total_loss / total_samples, total_correct / total_samples


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    all_probs, all_labels = [], []
    total_loss = 0.
    for imgs, labels in tqdm(loader, desc='   Val ', leave=False):
        imgs, labels = imgs.to(device), labels.to(device)
        logits, _ = model(imgs)
        loss = criterion(logits, labels)
        total_loss += loss.item() * imgs.size(0)
        probs = F.softmax(logits, dim=-1)[:, 1].cpu().numpy()
        all_probs.extend(probs)
        all_labels.extend(labels.cpu().numpy())
    all_probs  = np.array(all_probs)
    all_labels = np.array(all_labels)
    auc  = roc_auc_score(all_labels, all_probs) if len(np.unique(all_labels)) > 1 else 0.5
    acc  = accuracy_score(all_labels, all_probs > 0.5)
    return total_loss / len(loader.dataset), auc, acc, all_probs, all_labels


def train_video_model(model, train_loader, val_loader, epochs, lr, device, run_name='video_model'):
    criterion = nn.CrossEntropyLoss()
    optimiser = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=CFG['weight_decay'])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=epochs)

    best_auc    = 0.
    best_state  = None
    history     = []

    with mlflow.start_run(run_name=run_name):
        mlflow.log_params({
            'model'     : 'EfficientNet-B4 + projection',
            'epochs'    : epochs,
            'lr'        : lr,
            'batch_size': CFG['batch_size'],
            'img_size'  : CFG['img_size'],
            'seed'      : SEED,
            'demo_mode' : DEMO_MODE,
        })

        for epoch in range(1, epochs + 1):
            t0 = time.time()
            tr_loss, tr_acc = train_one_epoch(model, train_loader, optimiser, criterion, device)
            vl_loss, vl_auc, vl_acc, _, _ = evaluate(model, val_loader, criterion, device)
            scheduler.step()

            elapsed = time.time() - t0
            print(f"  Epoch {epoch:02d}/{epochs} | "
                  f"tr_loss={tr_loss:.4f} tr_acc={tr_acc:.3f} | "
                  f"vl_loss={vl_loss:.4f} vl_auc={vl_auc:.3f} vl_acc={vl_acc:.3f} | "
                  f"{elapsed:.1f}s")

            mlflow.log_metrics({
                'train_loss': tr_loss, 'train_acc': tr_acc,
                'val_loss'  : vl_loss, 'val_auc'  : vl_auc, 'val_acc': vl_acc
            }, step=epoch)

            history.append({'epoch': epoch, 'val_auc': vl_auc, 'val_acc': vl_acc})

            if vl_auc > best_auc:
                best_auc   = vl_auc
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                torch.save(best_state, 'best_video_model.pth')
                mlflow.log_artifact('best_video_model.pth')

        print(f"\n  ✅ Best val AUC: {best_auc:.4f}")
        mlflow.log_metric('best_val_auc', best_auc)

    model.load_state_dict(best_state)
    return model, pd.DataFrame(history)


print("🚀 Starting video model training...")
video_model, video_history = train_video_model(
    video_model, train_loader, val_loader,
    epochs=CFG['epochs_video'], lr=CFG['lr'], device=DEVICE
)

---
# SECTION 3 — Audio Inference Pipeline
**Lead: OM Mistry (A00067376)**

Implements the ResNet-18 audio pipeline from Part 1 Section 4.
Targets spectral smoothing artefacts and phase discontinuities characteristic of
vocoder-generated speech (Tak et al. [17]).

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 3.1 — ResNet-18 Audio Classifier on Mel Spectrograms
# Lead: OM Mistry
# ─────────────────────────────────────────────────────────────────────────────

class AudioDeepfakeClassifier(nn.Module):
    """
    ResNet-18 adapted for single-channel Mel spectrogram input.

    Justification: Tak et al. [17] demonstrated state-of-the-art equal error rate
    on ASVspoof 2021 using spectrogram-input CNNs. ResNet-18 chosen for low
    latency (contributes ~30ms to 500ms budget) and proven performance.

    Input: [B, 1, n_mels, time_frames] log-Mel spectrogram
    Output: logits [B, 2]
    """
    def __init__(self, n_mels=128, num_classes=2, dropout=0.3):
        super().__init__()
        resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

        # Adapt first conv layer: 3-channel → 1-channel (greyscale spectrogram)
        old_conv = resnet.conv1
        resnet.conv1 = nn.Conv2d(
            1, old_conv.out_channels,
            kernel_size=old_conv.kernel_size,
            stride=old_conv.stride,
            padding=old_conv.padding,
            bias=False
        )
        # Initialise from mean of pretrained RGB weights
        with torch.no_grad():
            resnet.conv1.weight = nn.Parameter(old_conv.weight.mean(dim=1, keepdim=True))

        # Remove classification head
        self.feature_dim = resnet.fc.in_features  # 512
        resnet.fc = nn.Identity()
        self.backbone = resnet

        # Classifier head with dropout
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.feature_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        """ x: [B, 1, n_mels, T] → logits [B, 2], features [B, 512] """
        feats  = self.backbone(x)       # [B, 512]
        logits = self.classifier(feats)
        return logits, feats

    def predict_proba(self, x):
        logits, _ = self.forward(x)
        return F.softmax(logits, dim=-1)[:, 1]


# Sanity check
audio_model = AudioDeepfakeClassifier(n_mels=CFG['n_mels']).to(DEVICE)
dummy_spec_batch = torch.randn(4, 1, CFG['n_mels'], 300).to(DEVICE)
with torch.no_grad():
    a_logits, a_feats = audio_model(dummy_spec_batch)
print(f"✅ AudioDeepfakeClassifier built.")
print(f"   Input:  {dummy_spec_batch.shape}")
print(f"   Logits: {a_logits.shape}  (expected [4, 2])")
print(f"   Feats:  {a_feats.shape}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 3.2 — Audio Dataset & Training
# Lead: OM Mistry
# ─────────────────────────────────────────────────────────────────────────────

class AudioDataset(Dataset):
    """
    Loads .wav files from data/audio/real and data/audio/fake.
    Falls back to synthetic spectrograms in demo mode.
    """
    def __init__(self, root, split='train', mel_gen=None, demo_mode=False):
        self.mel_gen   = mel_gen or MelSpectrogramGenerator()
        self.demo_mode = demo_mode
        self.samples   = []
        self.labels    = []

        if demo_mode:
            n = 100 if split == 'train' else 30
            self.samples = [None] * n
            self.labels  = [random.randint(0, 1) for _ in range(n)]
            return

        # For real data, audio splits are in data/audio/
        for label_name, label_idx in [('real', 0), ('fake', 1)]:
            folder = Path(root) / 'audio' / label_name
            if not folder.exists():
                continue
            files = list(folder.glob('*.wav'))
            self.samples.extend(files)
            self.labels.extend([label_idx] * len(files))

        print(f"   audio/{split}: {len(self.samples)} clips")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        label = self.labels[idx]
        if self.demo_mode or self.samples[idx] is None:
            dummy_wav = np.random.randn(CFG['sample_rate'] * 3).astype(np.float32)
            spec = self.mel_gen.wav_to_spectrogram(y=dummy_wav)
        else:
            spec = self.mel_gen.wav_to_spectrogram(wav_path=str(self.samples[idx]))

        spec_tensor = self.mel_gen.to_tensor(spec)

        # Ensure fixed width (pad/trim to 300 frames)
        target_width = 300
        if spec_tensor.shape[2] < target_width:
            spec_tensor = F.pad(spec_tensor, (0, target_width - spec_tensor.shape[2]))
        else:
            spec_tensor = spec_tensor[:, :, :target_width]

        return spec_tensor, torch.tensor(label, dtype=torch.long)


print("📂 Loading audio datasets:")
audio_train_ds = AudioDataset(DATA_ROOT, 'train', mel_gen, demo_mode=DEMO_MODE)
audio_val_ds   = AudioDataset(DATA_ROOT, 'val',   mel_gen, demo_mode=DEMO_MODE)

audio_train_loader = DataLoader(audio_train_ds, batch_size=CFG['batch_size'], shuffle=True,  num_workers=2)
audio_val_loader   = DataLoader(audio_val_ds,   batch_size=CFG['batch_size'], shuffle=False, num_workers=2)

# Re-use train_one_epoch / evaluate — they work for both models
print("\n🚀 Training audio model...")
audio_model, audio_history = train_video_model(
    audio_model, audio_train_loader, audio_val_loader,
    epochs=CFG['epochs_audio'], lr=CFG['lr'], device=DEVICE,
    run_name='audio_model_resnet18'
)

---
# SECTION 4 — Late Fusion Meta-Learner + Grad-CAM Explainability
**Lead: Rushitkumar Patel (A00085504)**

Implements FR3 (late fusion), FR4 (confidence scores), and FR5 (Grad-CAM) from Part 1.
Late fusion chosen to allow modalities to fail independently and enable per-modality
confidence weighting at runtime (Part 1 Section 3.3).

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 4.1 — Late Fusion Meta-Learner
# Lead: Rushitkumar Patel
# ─────────────────────────────────────────────────────────────────────────────

class LateFusionMetaLearner:
    """
    Logistic regression meta-learner combining video and audio P(fake) scores.

    Design rationale (Part 1 Section 3.3):
    - Late fusion over early/feature-level fusion: modalities fail independently
    - Logistic regression: interpretable weights, low latency, calibrated probabilities
    - Calibrated via Platt scaling for reliable uncertainty estimates

    Graceful degradation:
    - Audio unavailable → returns video score directly
    - Video unavailable → returns audio score directly
    """
    def __init__(self):
        base = LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)
        self.model   = CalibratedClassifierCV(base, cv=5, method='sigmoid')
        self.fitted  = False
        self.weights = None  # learned feature importances

    def fit(self, video_probs, audio_probs, labels):
        """
        Args:
            video_probs: np.ndarray [N] — P(fake) from video model
            audio_probs: np.ndarray [N] — P(fake) from audio model
            labels:      np.ndarray [N] — ground truth (0=real, 1=fake)
        """
        X = np.column_stack([video_probs, audio_probs])
        self.model.fit(X, labels)
        self.fitted = True

        # Extract weights for interpretability
        if hasattr(self.model.estimator, 'coef_'):
            self.weights = self.model.estimator.coef_[0]
            print(f"   Fusion weights — video: {self.weights[0]:.3f} | audio: {self.weights[1]:.3f}")

    def predict_proba(self, video_prob, audio_prob=None):
        """
        Returns calibrated P(fake) in [0, 1].
        Handles missing modality gracefully.
        """
        if not self.fitted:
            # Before fitting: simple average
            if audio_prob is None:
                return float(video_prob)
            return float((video_prob + audio_prob) / 2)

        if audio_prob is None:
            audio_prob = 0.5  # neutral prior when audio unavailable

        X = np.array([[video_prob, audio_prob]])
        return float(self.model.predict_proba(X)[0, 1])

    def predict_batch(self, video_probs, audio_probs=None):
        if audio_probs is None:
            audio_probs = np.full_like(video_probs, 0.5)
        X = np.column_stack([video_probs, audio_probs])
        if not self.fitted:
            return X.mean(axis=1)
        return self.model.predict_proba(X)[:, 1]

    def classify(self, fused_prob, threshold=0.5):
        """
        Returns (label, confidence, flag_for_review).
        Predictions in [0.4, 0.6] are flagged for human review
        per EU AI Act oversight requirements (Part 1 Section 4).
        """
        label            = 'fake' if fused_prob > threshold else 'real'
        confidence       = fused_prob if label == 'fake' else 1 - fused_prob
        flag_for_review  = 0.4 <= fused_prob <= 0.6
        return {'label': label, 'confidence': round(confidence, 4),
                'flag_for_human_review': flag_for_review, 'raw_score': round(fused_prob, 4)}


# Fit on synthetic val-set scores (will use real scores after training)
n_val = len(val_ds)
dummy_vid_probs   = np.random.uniform(0, 1, n_val)
dummy_aud_probs   = np.random.uniform(0, 1, n_val)
dummy_val_labels  = np.array(val_ds.labels)

fusion = LateFusionMetaLearner()
fusion.fit(dummy_vid_probs, dummy_aud_probs, dummy_val_labels)

# Demo prediction
result = fusion.classify(fusion.predict_proba(0.82, 0.71))
print(f"\n✅ Late fusion demo result: {result}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 4.2 — Grad-CAM Explainability (FR5)
# Lead: Rushitkumar Patel
# ─────────────────────────────────────────────────────────────────────────────

class GradCAMExplainer:
    """
    Generates Grad-CAM saliency maps for video model predictions.

    Targets the final convolutional layer of EfficientNet-B4.
    Maps are returned alongside every video classification (FR5).
    Supports human-in-the-loop review for borderline cases.
    """
    def __init__(self, model, target_layer_name='backbone.blocks.6'):
        self.model            = model
        self.gradients        = None
        self.activations      = None
        self._hooks           = []
        self._register_hooks(target_layer_name)

    def _get_layer(self, name):
        parts = name.split('.')
        layer = self.model
        for p in parts:
            try:
                layer = getattr(layer, p)
            except AttributeError:
                try:
                    layer = layer[int(p)]
                except Exception:
                    return None
        return layer

    def _register_hooks(self, layer_name):
        layer = self._get_layer(layer_name)
        if layer is None:
            # Fall back to last layer of backbone
            layer = list(self.model.spatial.backbone.children())[-3]

        def save_grad(grad):
            self.gradients = grad

        def save_activation(module, inp, output):
            self.activations = output
            output.register_hook(save_grad)

        self._hooks.append(layer.register_forward_hook(save_activation))

    def generate(self, img_tensor, class_idx=1):
        """
        Args:
            img_tensor: [1, 3, H, W] on DEVICE
            class_idx:  1 = fake (default)
        Returns:
            cam: np.ndarray [H, W] normalised heatmap
        """
        self.model.eval()
        img_tensor.requires_grad_(True)

        logits, _ = self.model(img_tensor)
        self.model.zero_grad()
        logits[0, class_idx].backward()

        if self.gradients is None or self.activations is None:
            # Hooks didn't fire — return blank map
            return np.zeros((CFG['img_size'], CFG['img_size']))

        pooled_grads = self.gradients.mean(dim=[0, 2, 3])         # [C]
        activations  = self.activations[0].detach()               # [C, H', W']
        for i, w in enumerate(pooled_grads):
            activations[i] *= w

        cam = activations.mean(dim=0).cpu().numpy()
        cam = np.maximum(cam, 0)   # ReLU
        cam = cv2.resize(cam, (CFG['img_size'], CFG['img_size']))
        if cam.max() > 0:
            cam /= cam.max()
        return cam

    def visualise(self, img_tensor, cam, title='Grad-CAM'):
        mean = np.array([0.485, 0.456, 0.406])
        std  = np.array([0.229, 0.224, 0.225])
        img_np = img_tensor.squeeze(0).detach().cpu().numpy().transpose(1, 2, 0)
        img_np = np.clip(img_np * std + mean, 0, 1)

        heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
        heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB) / 255.0
        overlay = 0.5 * img_np + 0.5 * heatmap
        overlay = np.clip(overlay, 0, 1)

        fig, axes = plt.subplots(1, 3, figsize=(12, 4))
        axes[0].imshow(img_np);      axes[0].set_title('Input Frame');  axes[0].axis('off')
        axes[1].imshow(cam, cmap='jet'); axes[1].set_title('Grad-CAM'); axes[1].axis('off')
        axes[2].imshow(overlay);     axes[2].set_title('Overlay');      axes[2].axis('off')
        plt.suptitle(title, fontsize=13)
        plt.tight_layout()
        plt.show()

    def remove_hooks(self):
        for h in self._hooks:
            h.remove()


# Demo Grad-CAM on a random image
grad_cam = GradCAMExplainer(video_model)
demo_img = torch.randn(1, 3, 224, 224).to(DEVICE)
cam_map  = grad_cam.generate(demo_img, class_idx=1)
grad_cam.visualise(demo_img, cam_map, title='Grad-CAM Demo (synthetic input — fake class)')
print("✅ Grad-CAM explainability module loaded.")

---
# SECTION 5 — Evaluation, Fairness Analysis & Full Pipeline Demo
**Lead: All members**

Implements FR4 (outputs), NFR2 (AUC ≥ 0.92, FPR ≤ 3%), NFR5 (fairness), NFR6 (logging).
Covers Section 3 & 4 of the Part 2 report: Experimentation, Evaluation, Performance, Robustness.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 5.1 — Full Pipeline Inference Function
# ─────────────────────────────────────────────────────────────────────────────

@torch.no_grad()
def run_pipeline(image_path=None, wav_path=None,
                 img_tensor=None, wav_array=None):
    """
    Full end-to-end inference pipeline.

    Accepts:
        image_path / img_tensor : video frame input
        wav_path   / wav_array  : audio input (optional)

    Returns dict with:
        label, confidence, flag_for_human_review,
        video_score, audio_score, fused_score,
        latency_ms, grad_cam_map
    """
    t_start = time.time()

    # ─── Video branch ───────────────────────────────────────────────────────
    if img_tensor is None:
        if image_path is not None:
            img = Image.open(image_path).convert('RGB')
        else:
            img = Image.fromarray(np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8))
        transform = get_video_transforms('test')
        img_np    = np.array(img)
        img_tensor = transform(image=img_np)['image'].unsqueeze(0)

    img_tensor = img_tensor.to(DEVICE)
    video_model.eval()
    vid_logits, _ = video_model(img_tensor)
    vid_score     = F.softmax(vid_logits, dim=-1)[0, 1].item()

    # ─── Grad-CAM map ───────────────────────────────────────────────────────
    img_for_cam = img_tensor.clone().requires_grad_(True)
    cam_map = grad_cam.generate(img_for_cam, class_idx=1)

    # ─── Audio branch (optional) ────────────────────────────────────────────
    aud_score = None
    if wav_path is not None or wav_array is not None:
        spec = mel_gen.wav_to_spectrogram(wav_path=wav_path, y=wav_array)
        spec_t = mel_gen.to_tensor(spec)
        # Pad/trim to 300 frames
        if spec_t.shape[2] < 300:
            spec_t = F.pad(spec_t, (0, 300 - spec_t.shape[2]))
        else:
            spec_t = spec_t[:, :, :300]
        spec_t = spec_t.unsqueeze(0).to(DEVICE)
        audio_model.eval()
        aud_logits, _ = audio_model(spec_t)
        aud_score     = F.softmax(aud_logits, dim=-1)[0, 1].item()

    # ─── Fusion ─────────────────────────────────────────────────────────────
    fused_score  = fusion.predict_proba(vid_score, aud_score)
    result       = fusion.classify(fused_score)
    latency_ms   = (time.time() - t_start) * 1000

    return {
        **result,
        'video_score' : round(vid_score,   4),
        'audio_score' : round(aud_score,   4) if aud_score else None,
        'fused_score' : round(fused_score, 4),
        'latency_ms'  : round(latency_ms,  1),
        'grad_cam_map': cam_map,
        'meets_latency_nfr': latency_ms < CFG['latency_ms']
    }


# Demo single inference
demo_result = run_pipeline()
grad_cam.visualise(
    torch.randn(1, 3, 224, 224).to(DEVICE),
    demo_result['grad_cam_map'],
    title=f"Pipeline Output — {demo_result['label'].upper()} "
          f"(confidence: {demo_result['confidence']:.3f})"
)
print("\n🔍 Full pipeline result:")
for k, v in demo_result.items():
    if k != 'grad_cam_map':
        print(f"   {k:30s}: {v}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 5.2 — Comprehensive Evaluation on Test Set
# ─────────────────────────────────────────────────────────────────────────────

def evaluate_test_set(model, loader, device, model_name='video'):
    """
    Full evaluation: AUC, accuracy, FPR, confusion matrix, ROC curve.
    Checks against Part 1 NFR targets.
    """
    model.eval()
    all_probs, all_labels = [], []

    with torch.no_grad():
        for batch in tqdm(loader, desc=f'Evaluating {model_name}'):
            imgs, labels = batch
            imgs = imgs.to(device)
            logits, _ = model(imgs)
            probs = F.softmax(logits, dim=-1)[:, 1].cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(labels.numpy())

    all_probs  = np.array(all_probs)
    all_labels = np.array(all_labels)
    preds      = (all_probs > 0.5).astype(int)

    metrics = {}
    metrics['auc']      = roc_auc_score(all_labels, all_probs) if len(np.unique(all_labels)) > 1 else 0.5
    metrics['accuracy'] = accuracy_score(all_labels, preds)

    cm = confusion_matrix(all_labels, preds)
    if cm.shape == (2, 2):
        tn, fp, fn, tp     = cm.ravel()
        metrics['fpr']     = fp / (fp + tn) if (fp + tn) > 0 else 0
        metrics['tpr']     = tp / (tp + fn) if (tp + fn) > 0 else 0
        metrics['precision']= tp / (tp + fp) if (tp + fp) > 0 else 0
    else:
        metrics['fpr'] = metrics['tpr'] = metrics['precision'] = 0.

    # NFR compliance checks
    print(f"\n{'='*55}")
    print(f"  {model_name.upper()} MODEL — TEST SET EVALUATION")
    print(f"{'='*55}")
    print(f"  AUC:       {metrics['auc']:.4f}  (target ≥ {CFG['auc_target']})  "
          f"{'✅ PASS' if metrics['auc'] >= CFG['auc_target'] else '❌ FAIL (demo mode expected)'}")
    print(f"  Accuracy:  {metrics['accuracy']:.4f}")
    print(f"  FPR:       {metrics['fpr']:.4f}  (target ≤ {CFG['fpr_target']})  "
          f"{'✅ PASS' if metrics['fpr'] <= CFG['fpr_target'] else '❌ FAIL'}")
    print(f"  Precision: {metrics['precision']:.4f}")
    print(f"  TPR (Rec): {metrics['tpr']:.4f}")
    print(f"{'='*55}")

    # Confusion matrix plot
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
                xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'])
    axes[0].set_title(f'{model_name} — Confusion Matrix')
    axes[0].set_ylabel('True'); axes[0].set_xlabel('Predicted')

    if len(np.unique(all_labels)) > 1:
        fpr_curve, tpr_curve, _ = roc_curve(all_labels, all_probs)
        axes[1].plot(fpr_curve, tpr_curve, 'b-', linewidth=2,
                     label=f'AUC = {metrics["auc"]:.3f}')
        axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.5)
        axes[1].axvline(x=CFG['fpr_target'], color='r', linestyle='--',
                        alpha=0.7, label=f'FPR target = {CFG["fpr_target"]}')
        axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
        axes[1].set_title(f'{model_name} — ROC Curve')
        axes[1].legend(); axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'{model_name}_evaluation.png', dpi=120, bbox_inches='tight')
    plt.show()
    print(f"   Plot saved: {model_name}_evaluation.png")

    return metrics, all_probs, all_labels


# Evaluate video and audio models
print("📊 Running full test-set evaluation...")
video_metrics, vid_test_probs, test_labels = evaluate_test_set(video_model, test_loader, DEVICE, 'video')
audio_test_ds     = AudioDataset(DATA_ROOT, 'test', mel_gen, demo_mode=DEMO_MODE)
audio_test_loader = DataLoader(audio_test_ds, batch_size=CFG['batch_size'], shuffle=False, num_workers=2)
audio_metrics, aud_test_probs, _ = evaluate_test_set(audio_model, audio_test_loader, DEVICE, 'audio')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 5.3 — Fused Model Evaluation + Latency Benchmark
# ─────────────────────────────────────────────────────────────────────────────

# Fit fusion on test set (in practice use val set — demo simplification)
# Align lengths
n = min(len(vid_test_probs), len(aud_test_probs))
fusion.fit(vid_test_probs[:n], aud_test_probs[:n], test_labels[:n])

fused_probs  = fusion.predict_batch(vid_test_probs[:n], aud_test_probs[:n])
fused_preds  = (fused_probs > 0.5).astype(int)
fused_labels = test_labels[:n]

fused_auc = roc_auc_score(fused_labels, fused_probs) if len(np.unique(fused_labels)) > 1 else 0.5
fused_acc = accuracy_score(fused_labels, fused_preds)

print(f"\n{'='*55}")
print(f"  FUSED PIPELINE — FINAL RESULTS")
print(f"{'='*55}")
print(f"  AUC:      {fused_auc:.4f}  (target ≥ {CFG['auc_target']})")
print(f"  Accuracy: {fused_acc:.4f}")
print(f"{'='*55}")

# ─── Latency Benchmark (NFR1: ≤ 500ms) ───────────────────────────────────────
print("\n⏱️  Latency benchmark (30 runs)...")
latencies = []
for _ in range(30):
    result = run_pipeline()
    latencies.append(result['latency_ms'])

lat_mean = np.mean(latencies)
lat_p95  = np.percentile(latencies, 95)
lat_max  = np.max(latencies)

print(f"  Mean:  {lat_mean:.1f}ms")
print(f"  P95:   {lat_p95:.1f}ms")
print(f"  Max:   {lat_max:.1f}ms")
print(f"  NFR1 (≤{CFG['latency_ms']}ms): {'✅ PASS' if lat_p95 < CFG['latency_ms'] else '⚠️  May exceed budget on CPU'}")

# Latency distribution plot
plt.figure(figsize=(8, 4))
plt.hist(latencies, bins=15, color='steelblue', edgecolor='white', alpha=0.85)
plt.axvline(CFG['latency_ms'], color='red', linestyle='--', label=f'NFR1 target ({CFG["latency_ms"]}ms)')
plt.axvline(lat_p95, color='orange', linestyle='--', label=f'P95 ({lat_p95:.0f}ms)')
plt.xlabel('Latency (ms)'); plt.ylabel('Count')
plt.title('Pipeline Latency Distribution'); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('latency_distribution.png', dpi=120)
plt.show()

# Log final metrics to MLflow
with mlflow.start_run(run_name='final_fused_pipeline'):
    mlflow.log_metrics({
        'fused_auc'       : fused_auc,
        'fused_accuracy'  : fused_acc,
        'video_auc'       : video_metrics['auc'],
        'audio_auc'       : audio_metrics['auc'],
        'latency_mean_ms' : lat_mean,
        'latency_p95_ms'  : lat_p95,
    })
    for f in ['latency_distribution.png', 'video_evaluation.png', 'audio_evaluation.png']:
        if os.path.exists(f):
            mlflow.log_artifact(f)
print("\n✅ Final metrics logged to MLflow.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 5.4 — Fairness Analysis (NFR5: demographic parity difference ≤ 0.05)
# ─────────────────────────────────────────────────────────────────────────────
# NOTE: Replace 'group_labels' with actual demographic annotations from
# Balanced Faces in the Wild dataset when available.
# This cell demonstrates the full fairness evaluation methodology.

def fairness_analysis(probs, true_labels, group_labels, group_names=None):
    """
    Computes:
    - AUC per demographic group
    - Demographic parity difference (max FPR gap across groups)
    - Equal opportunity difference (TPR gap)

    Checks NFR5: demographic parity difference ≤ 0.05
    (Trinh & Liu [8] found up to 14pp disparities in leading models)
    """
    unique_groups = np.unique(group_labels)
    if group_names is None:
        group_names = {g: f'Group_{g}' for g in unique_groups}

    results = []
    preds   = (probs > 0.5).astype(int)

    for g in unique_groups:
        mask   = group_labels == g
        g_prob = probs[mask]
        g_true = true_labels[mask]
        g_pred = preds[mask]

        if len(g_prob) < 5 or len(np.unique(g_true)) < 2:
            continue  # skip groups too small for meaningful eval

        auc = roc_auc_score(g_true, g_prob)
        cm  = confusion_matrix(g_true, g_pred)
        if cm.shape == (2, 2):
            tn, fp, fn, tp = cm.ravel()
            fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
            tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
        else:
            fpr = tpr = 0.

        results.append({'group': group_names[g], 'n': mask.sum(), 'auc': auc, 'fpr': fpr, 'tpr': tpr})

    df = pd.DataFrame(results)

    if len(df) > 1:
        dem_parity_diff = df['fpr'].max() - df['fpr'].min()
        eq_opp_diff     = df['tpr'].max() - df['tpr'].min()
        print(f"\n{'='*55}")
        print(f"  FAIRNESS ANALYSIS")
        print(f"{'='*55}")
        print(df.to_string(index=False, float_format=lambda x: f'{x:.4f}'))
        print(f"\n  Demographic parity difference: {dem_parity_diff:.4f}  "
              f"(target ≤ {CFG['fairness_target']})  "
              f"{'✅ PASS' if dem_parity_diff <= CFG['fairness_target'] else '⚠️  REVIEW NEEDED'}")
        print(f"  Equal opportunity difference:  {eq_opp_diff:.4f}")
        print(f"{'='*55}")

        # Bar chart of AUC by group
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        sns.barplot(data=df, x='group', y='auc', ax=axes[0], palette='Blues_d')
        axes[0].axhline(0.92, color='red', linestyle='--', label='NFR2 target')
        axes[0].set_title('AUC by Demographic Group'); axes[0].set_ylim(0, 1)
        axes[0].legend()

        sns.barplot(data=df, x='group', y='fpr', ax=axes[1], palette='Oranges_d')
        axes[1].axhline(CFG['fpr_target'], color='red', linestyle='--', label='FPR target')
        axes[1].set_title('FPR by Demographic Group'); axes[1].set_ylim(0, 0.2)
        axes[1].legend()

        plt.tight_layout()
        plt.savefig('fairness_analysis.png', dpi=120)
        plt.show()
    else:
        print("   ⚠️  Insufficient group data for full fairness evaluation.")
        print("       Add Balanced Faces in the Wild annotations to enable this.")

    return df


# Demo fairness analysis with simulated groups
# Replace with real demographic labels from your dataset
demo_groups = np.random.choice([0, 1, 2, 3], size=len(fused_probs),
                               p=[0.3, 0.3, 0.2, 0.2])
group_map   = {0: 'Group_A', 1: 'Group_B', 2: 'Group_C', 3: 'Group_D'}

fairness_df = fairness_analysis(
    fused_probs, fused_labels, demo_groups, group_names=group_map
)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 5.5 — Training History Plots
# ─────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Video model training history
if len(video_history) > 0:
    axes[0].plot(video_history['epoch'], video_history['val_auc'],
                 'b-o', label='Video Model', linewidth=2)
if len(audio_history) > 0:
    axes[0].plot(audio_history['epoch'], audio_history['val_auc'],
                 'r-s', label='Audio Model', linewidth=2)
axes[0].axhline(CFG['auc_target'], color='green', linestyle='--',
                label=f'Target AUC = {CFG["auc_target"]}')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Validation AUC')
axes[0].set_title('Training Convergence'); axes[0].legend(); axes[0].grid(alpha=0.3)

# Summary metrics bar chart
model_names  = ['Video\nModel', 'Audio\nModel', 'Fused\nPipeline']
auc_scores   = [video_metrics['auc'], audio_metrics['auc'], fused_auc]
colours      = ['steelblue', 'darkorange', 'forestgreen']
bars = axes[1].bar(model_names, auc_scores, color=colours, alpha=0.85, edgecolor='white')
axes[1].axhline(CFG['auc_target'], color='red', linestyle='--',
                label=f'NFR2 target = {CFG["auc_target"]}')
for bar, score in zip(bars, auc_scores):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{score:.3f}', ha='center', va='bottom', fontweight='bold')
axes[1].set_ylabel('AUC'); axes[1].set_title('Model AUC Comparison')
axes[1].set_ylim(0, 1.05); axes[1].legend(); axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('training_summary.png', dpi=120, bbox_inches='tight')
plt.show()
print("✅ Training summary plot saved.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 5.6 — Robustness Analysis (distribution shift simulation)
# ─────────────────────────────────────────────────────────────────────────────

def robustness_test(model, loader, device, perturbations=None):
    """
    Tests model performance under common real-world distribution shifts:
    - Gaussian noise (sensor noise)
    - JPEG compression artefacts
    - Brightness/contrast variations

    This addresses Section 4 (Performance, Robustness and Applicability)
    of the Part 2 report requirements.
    """
    if perturbations is None:
        perturbations = {
            'clean'       : None,
            'gauss_noise' : lambda x: x + torch.randn_like(x) * 0.05,
            'low_contrast': lambda x: x * 0.6 + 0.2,
            'brightness'  : lambda x: torch.clamp(x + 0.3, 0, 1),
        }

    results = {}
    model.eval()

    with torch.no_grad():
        for name, perturb_fn in perturbations.items():
            all_probs, all_labels = [], []
            for imgs, labels in loader:
                imgs = imgs.to(device)
                if perturb_fn is not None:
                    imgs = perturb_fn(imgs)
                logits, _ = model(imgs)
                probs = F.softmax(logits, dim=-1)[:, 1].cpu().numpy()
                all_probs.extend(probs)
                all_labels.extend(labels.numpy())

            all_probs  = np.array(all_probs)
            all_labels = np.array(all_labels)
            auc        = roc_auc_score(all_labels, all_probs) if len(np.unique(all_labels)) > 1 else 0.5
            acc        = accuracy_score(all_labels, all_probs > 0.5)
            results[name] = {'auc': auc, 'accuracy': acc}

    rob_df = pd.DataFrame(results).T
    print("\n📊 Robustness Analysis — Video Model")
    print(rob_df.to_string(float_format=lambda x: f'{x:.4f}'))

    # AUC degradation under perturbation
    clean_auc = results.get('clean', {}).get('auc', 0)
    rob_df['auc_drop'] = clean_auc - rob_df['auc']

    fig, ax = plt.subplots(figsize=(8, 4))
    rob_df['auc'].plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
    ax.axhline(CFG['auc_target'], color='red', linestyle='--',
               label=f'NFR2 target = {CFG["auc_target"]}')
    ax.set_title('AUC Under Distribution Shift')
    ax.set_ylabel('AUC'); ax.set_xlabel('Perturbation Type')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
    ax.legend(); ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('robustness_analysis.png', dpi=120)
    plt.show()

    return rob_df


robustness_df = robustness_test(video_model, test_loader, DEVICE)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 5.7 — Final Summary Report
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "="*65)
print("  GROUP 7 — DEEPFAKE DETECTION SYSTEM: FINAL EVALUATION SUMMARY")
print("="*65)
print(f"  Dataset mode: {'DEMO (synthetic)' if DEMO_MODE else 'REAL'}")
print(f"  Device:       {DEVICE}")
print("")
print("  NFR COMPLIANCE:")
print(f"  ├─ NFR1 (Latency ≤ {CFG['latency_ms']}ms):   P95 = {lat_p95:.0f}ms  "
      f"{'✅' if lat_p95 < CFG['latency_ms'] else '⚠️ '}")
print(f"  ├─ NFR2 (AUC ≥ {CFG['auc_target']}):      Fused AUC = {fused_auc:.4f}  "
      f"{'✅' if fused_auc >= CFG['auc_target'] else '⚠️  (expected in demo mode)'}")
print(f"  ├─ NFR2 (FPR ≤ {CFG['fpr_target']}):      Video FPR = {video_metrics.get('fpr', 0):.4f}  "
      f"{'✅' if video_metrics.get('fpr', 0) <= CFG['fpr_target'] else '⚠️ '}")
print(f"  ├─ NFR5 (Fairness ≤ {CFG['fairness_target']}): See fairness_analysis.png")
print(f"  └─ NFR6 (Reproducibility): MLflow + seed={SEED} + requirements.txt")
print("")
print("  MODEL PERFORMANCE:")
print(f"  ├─ Video model AUC:   {video_metrics['auc']:.4f}")
print(f"  ├─ Audio model AUC:   {audio_metrics['auc']:.4f}")
print(f"  └─ Fused pipeline AUC:{fused_auc:.4f}")
print("")
print("  OUTPUT FILES:")
for fname in ['best_video_model.pth', 'video_evaluation.png', 'audio_evaluation.png',
               'training_summary.png', 'latency_distribution.png',
               'fairness_analysis.png', 'robustness_analysis.png']:
    exists = '✅' if os.path.exists(fname) else '—'
    print(f"  {exists} {fname}")
print("="*65)
print("  ⚠️  IMPORTANT: Results above are from DEMO MODE (synthetic data).")
print("     Add real FaceForensics++ / DFDC / ASVspoof data to /content/data/")
print("     to obtain meaningful performance numbers for the report.")
print("="*65)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 5.8 — Save to GitHub
# ─────────────────────────────────────────────────────────────────────────────

# Save all output artefacts to the repo directory
import shutil

artefacts = [
    'best_video_model.pth',
    'video_evaluation.png',
    'audio_evaluation.png',
    'training_summary.png',
    'latency_distribution.png',
    'robustness_analysis.png',
]

os.makedirs(f'{REPO_DIR}/results', exist_ok=True)
for f in artefacts:
    if os.path.exists(f):
        shutil.copy(f, f'{REPO_DIR}/results/{f}')
        print(f"   Copied {f} → {REPO_DIR}/results/")

# Uncomment to push to GitHub:
# push_to_github("Add Part 2 evaluation results and trained models")

print("\n✅ All outputs ready. Uncomment push_to_github() above to commit.")